# LAB 01 — Experiments

Notebook cho ba phần đầu của LAB 01: setup dataset, kiểm tra sparse representation và vocabulary inspection.

## 1. Setup and Dataset

Corpus được đọc từ file JSON Lines nén gzip. Mỗi dòng cần có trường `text`; các trường khác được giữ lại trong metadata. Cell cấu hình tự tìm dataset trong `lab01/data`, `Downloads` của user hoặc thư mục hiện tại. Nếu không tìm thấy, hãy đặt `DATA_PATH` trong cell này tới corpus 30K được cung cấp.

In [ ]:
from __future__ import annotations

import gzip
import json
from pathlib import Path

import numpy as np
import pandas as pd

DATA_FILENAME = "c4-train.00000-of-01024-30K.json.gz"

def find_repo_root() -> Path:
    """Find the repository root from the notebook working directory."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "lab01").is_dir() and (candidate / "README.md").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = find_repo_root()
DATA_CANDIDATES = [
    REPO_ROOT / "lab01" / "data" / DATA_FILENAME,
    REPO_ROOT / "data" / DATA_FILENAME,
    Path.home() / "Downloads" / DATA_FILENAME,
    Path.cwd() / DATA_FILENAME,
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    DATA_AVAILABLE = False
    records = []
    documents = []
    print("Dataset not found. Set DATA_PATH to the provided 30K corpus.")
else:
    DATA_AVAILABLE = True
    with gzip.open(DATA_PATH, "rt", encoding="utf-8") as stream:
        records = [json.loads(line) for line in stream if line.strip()]
    documents = [record.get("text", "") for record in records]
    print(f"Dataset: {DATA_PATH}")
    print(f"Number of documents: {len(documents):,}")
    print(f"Empty documents: {sum(not text.strip() for text in documents):,}")
    display(pd.DataFrame(records[:3]))


## 2. Experiment 1 — Sparse Representation

Pipeline: raw documents → tokenizer → CountVectorizer → normalized TF → IDF → TF-IDF matrix. The large matrix remains sparse throughout.

In [ ]:
try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
    from sklearn.preprocessing import normalize
    SKLEARN_AVAILABLE = True
except ImportError as error:
    SKLEARN_AVAILABLE = False
    print(f"scikit-learn is required for Experiment 1: {error}")

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    vectorizer = CountVectorizer()
    count_matrix = vectorizer.fit_transform(documents)
    tf_matrix = normalize(count_matrix, norm="l1", axis=1, copy=True)

    # smooth_idf=False uses unsmoothed document frequencies, but
    # sklearn's idf convention still includes the additive +1 offset.
    tfidf_transformer = TfidfTransformer(
        norm=None,
        use_idf=True,
        smooth_idf=False,
    )
    tfidf_matrix = tfidf_transformer.fit_transform(tf_matrix)
    feature_names = vectorizer.get_feature_names_out()

    number_of_documents, vocabulary_size = tfidf_matrix.shape
    nnz = tfidf_matrix.nnz
    total_entries = number_of_documents * vocabulary_size
    sparsity = 1 - nnz / total_entries if total_entries else 0.0

    summary = pd.DataFrame({
        "metric": [
            "Number of documents",
            "Vocabulary size",
            "TF-IDF matrix shape",
            "Non-zero entries (nnz)",
            "Sparsity",
        ],
        "value": [
            number_of_documents,
            vocabulary_size,
            tfidf_matrix.shape,
            nnz,
            sparsity,
        ],
    })
    display(summary)
    print(f"Sparse matrix type: {type(tfidf_matrix).__name__}")
elif not DATA_AVAILABLE:
    print("Experiment 1 skipped because the dataset is not available.")
else:
    print("Experiment 1 skipped because scikit-learn is not available.")


## 3. Vocabulary Inspection

The tables below use the feature ordering returned by the fitted vectorizer. Document frequency means the number of documents containing a term, not the total number of occurrences.

In [ ]:
def get_top_df_terms(count_matrix, feature_names, top_k=20):
    """Return terms ranked by document frequency."""
    document_frequency = np.asarray(count_matrix.getnnz(axis=0)).ravel()
    table = pd.DataFrame({
        "term": feature_names,
        "document_frequency": document_frequency,
    })
    return table.sort_values("document_frequency", ascending=False).head(top_k).reset_index(drop=True)

def get_top_idf_terms(feature_names, idf, top_k=20):
    """Return terms ranked by the fitted transformer IDF values."""
    table = pd.DataFrame({"term": feature_names, "idf": idf})
    return table.sort_values("idf", ascending=False).head(top_k).reset_index(drop=True)

def get_top_tfidf_terms(tfidf_matrix, feature_names, document_index, top_k=20):
    """Return non-zero TF-IDF terms for one selected document."""
    row = tfidf_matrix.getrow(document_index)
    table = pd.DataFrame({
        "term": feature_names[row.indices],
        "tfidf": row.data,
    })
    return table.sort_values("tfidf", ascending=False).head(top_k).reset_index(drop=True)

if DATA_AVAILABLE and SKLEARN_AVAILABLE:
    top_df_terms = get_top_df_terms(count_matrix, feature_names)
    top_idf_terms = get_top_idf_terms(feature_names, tfidf_transformer.idf_)

    SELECTED_DOC_INDEX = 0
    top_tfidf_terms = get_top_tfidf_terms(
        tfidf_matrix, feature_names, SELECTED_DOC_INDEX
    )

    print("Top 20 terms by document frequency")
    display(top_df_terms)
    print("Top 20 terms by IDF")
    display(top_idf_terms)
    print(f"Top TF-IDF terms in document {SELECTED_DOC_INDEX}")
    print(documents[SELECTED_DOC_INDEX][:500])
    display(top_tfidf_terms)
elif not DATA_AVAILABLE:
    print("Vocabulary inspection skipped because the dataset is not available.")
else:
    print("Vocabulary inspection skipped because scikit-learn is not available.")


### Student analysis

TODO:
- Compare the three term lists.
- Does a frequent corpus term necessarily have high TF-IDF?
- Does a high-IDF term necessarily have high TF-IDF in every document?

## 4. Experiment 2 — Preprocessing Ablation

TODO: implement Pipeline A, Pipeline B and Pipeline C only after the first three sections are reviewed.

## 5. Document Search

TODO: build the query vector and rank documents after preprocessing ablation.

## 6. Evaluation

TODO: add student-provided relevance labels and compute Precision@5, Recall@5 and MRR.

## 7. Error Analysis

TODO: inspect selected good and poor queries after the evaluation set is provided.